# 📚 Bookcase Digitization - Huấn luyện mô hình YOLOv5x6
**Team:** Tú - Lâm - Ngọc

Notebook này được thiết kế để tự động hóa toàn bộ quá trình thiết lập và huấn luyện mô hình nhận diện gáy sách.

### Bước 1: Kiểm tra GPU và Cài đặt môi trường

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Xóa bản cũ (nếu có) và Clone lại bản mới nhất đã có data.yaml
%cd /content/
!rm -rf bookcase-digitization
!git clone https://github.com/pie-12/bookcase-digitization.git
%cd bookcase-digitization

# 3. Cài đặt thư viện
!pip install -r requirements.txt
!pip install -q craft-text-detector vietocr==0.3.5

# 4. Clone YOLOv5 chính thức
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt

### Bước 2: Huấn luyện (Training)
Chúng ta sử dụng đường dẫn tuyệt đối để tránh lỗi File Not Found.

In [ ]:
import os
%env WANDB_MODE=disabled

# Đường dẫn tuyệt đối đến data.yaml
DATA_YAML = "/content/bookcase-digitization/data.yaml"

# Lệnh Train
!python train.py --img 720 --batch 8 --epochs 100 --data {DATA_YAML} --weights yolov5x6.pt --project ../runs/train --name bookcase_model

### Bước 3: Xem kết quả

In [ ]:
# Thử nghiệm trên ảnh mẫu
!python detect.py --weights ../runs/train/bookcase_model/weights/best.pt --img 720 --conf 0.25 --source ../data_test/1624445642850.jpg

from IPython.display import Image
import glob
latest_exp = max(glob.glob('../runs/detect/exp*'), key=os.path.getmtime)
Image(filename=os.path.join(latest_exp, '1624445642850.jpg'))

### Bước 4: Tải Model về

In [ ]:
from google.colab import files
!cp ../runs/train/bookcase_model/weights/best.pt /content/best.pt
files.download('/content/best.pt')